# Native Kaggle resource and CPU dispatch probe
Uses the documented notebook resource integration with existing permissions. Forbidden HTTP status/download methods are not retried; no credentials or permissions are created.

In [ ]:
import hashlib
import json
import os
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

base = Path("/kaggle/working/hung-native-api-probe-v4")
base.mkdir(exist_ok=False)
os.environ["KAGGLEHUB_CACHE"] = str(base / "cache")
assert os.environ.get("DISABLE_KAGGLE_CACHE", "").lower() not in ("1", "true", "t"), (
    "Native documented resolver required"
)

In [ ]:
import kagglehub
from kagglehub.clients import build_kaggle_client
from kagglesdk.kernels.types.kernels_api_service import (
    ApiGetAcceleratorQuotaStatisticsRequest,
    ApiGetKernelSessionStatusRequest,
    ApiSaveKernelRequest,
)
from kagglesdk.kernels.types.kernels_enums import KernelWorkerStatus

handle = "hungluongduc/hung-e2e-api-storage-probe-20261027-v3/versions/1"
expected = "ec222656ce333085a7d894acc3ed9675031a22a2703a86d5e6e777aaa934a5aa"
started = time.perf_counter()
remote = Path(kagglehub.dataset_download(handle, path="probe.zip.bin", force_download=True))
if remote.is_dir():
    remote = remote / "probe.zip.bin"
assert remote.stat().st_size == 303
assert hashlib.sha256(remote.read_bytes()).hexdigest() == expected
with zipfile.ZipFile(remote) as archive:
    assert archive.testzip() is None and archive.namelist() == ["probe.json"]
native = {
    "passed": True,
    "dataset": handle,
    "sha256": expected,
    "bytes": 303,
    "crc": "passed",
    "seconds": time.perf_counter() - started,
    "readback_path": str(remote),
    "scope": "Documented native notebook resource mount; HTTP DownloadDataset403 is not retried",
}
with (base / "native-readback.json").open("x") as stream:
    json.dump(native, stream, indent=2)
print(json.dumps({"native_readback": native}), flush=True)
payload = {
    "schema": "hung-cpu-api-dispatch-probe-v1",
    "model_training": False,
    "parent_code_sha": "44b8532959d76fb20691881539f991ca44983727",
}
payload_text = json.dumps(payload, sort_keys=True) + "\n"
payload_sha = hashlib.sha256(payload_text.encode()).hexdigest()
child_source = (
    'from pathlib import Path\nPath("/kaggle/working/completion.json").write_text('
    + repr(payload_text)
    + ',encoding="utf-8")\nprint("CPU_DISPATCH_PROBE_COMPLETE")\n'
)
request = ApiSaveKernelRequest()
request.slug = "hungluongduc/hung-api-dispatch-probe-20261027-v1"
request.new_title = "hung-api-dispatch-probe-20261027-v1"
request.text = child_source
request.language = "python"
request.kernel_type = "script"
request.is_private = True
request.enable_gpu = False
request.enable_internet = False
request.session_timeout_seconds = 300
with build_kaggle_client() as client:
    saved = client.kernels.kernels_api_client.save_kernel(request)
    assert not saved.error and saved.version_number == 1, "Unexpected child submission/version"
    submitted = {
        "url": saved.url,
        "ref": saved.ref,
        "version": saved.version_number,
        "expected_completion_sha256": payload_sha,
        "code_sha256": hashlib.sha256(child_source.encode()).hexdigest(),
        "gpu": False,
        "is_private_requested": True,
    }
    with (base / "dispatch-submitted.json").open("x") as stream:
        json.dump(submitted, stream, indent=2)
    print(json.dumps({"cpu_child_submitted": submitted}), flush=True)
    status_request = ApiGetKernelSessionStatusRequest()
    status_request.user_name = "hungluongduc"
    status_request.kernel_slug = "hung-api-dispatch-probe-20261027-v1"
    for attempt in range(60):
        status = client.kernels.kernels_api_client.get_kernel_session_status(status_request)
        if status.status == KernelWorkerStatus.COMPLETE:
            break
        assert status.status in (KernelWorkerStatus.QUEUED, KernelWorkerStatus.RUNNING), (
            status.status.name,
            status.failure_message,
        )
        time.sleep(10)
    else:
        raise TimeoutError("CPU child did not complete; no successful dispatch claim")
completion = Path(
    kagglehub.notebook_output_download(
        "hungluongduc/hung-api-dispatch-probe-20261027-v1/versions/1",
        path="completion.json",
        force_download=True,
    )
)
if completion.is_dir():
    completion = completion / "completion.json"
assert hashlib.sha256(completion.read_bytes()).hexdigest() == payload_sha
with build_kaggle_client() as client:
    gpu = client.kernels.kernels_api_client.get_accelerator_quota_statistics(
        ApiGetAcceleratorQuotaStatisticsRequest()
    ).gpu_quota
    assert gpu is not None and all(
        v is not None for v in (gpu.total_time_allowed, gpu.time_used, gpu.time_reserved)
    )
    quota = {
        "observed_at_utc": datetime.now(timezone.utc).isoformat(),
        "total_seconds": gpu.total_time_allowed.total_seconds(),
        "used_seconds": gpu.time_used.total_seconds(),
        "reserved_seconds": gpu.time_reserved.total_seconds(),
        "pay_to_scale_enabled": gpu.is_pay_to_scale_enabled,
    }
result = {
    "passed": True,
    "native_readback": native,
    "cpu_dispatch": submitted,
    "completion_sha256": payload_sha,
    "actual_cpu_child_complete": True,
    "quota": quota,
    "kagglehub_version": kagglehub.__version__,
    "scope": "Private native resource readback plus CPU-only SDK dispatch/completion proof; no training or Final",
}
with (base / "result.json").open("x") as stream:
    json.dump(result, stream, indent=2)
print(json.dumps(result), flush=True)